<a href="https://colab.research.google.com/github/SIRLLON/PB/blob/main/tp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP 3 (TP3)
**Aluno:** Sirllon
**Plataforma:** Google Colab Notebook  
Contém a implementação de Árvores de Busca Binária (BST), computação paralela com OpenMP e Tries direcionadas a roteamento IP (LPM).

### Grupo 1: Árvore Binária de Busca (BST)
Implementação de inserção, busca, remoção, cálculo de altura e testes de tempo de execução.

In [ ]:
import time
import random

class BSTNode:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None

class BST:
    def __init__(self):
        self.root = None

    def inserir(self, key):
        # Insere valor na BST
        self.root = self._inserir(self.root, key)

    def _inserir(self, root, key):
        if not root: return BSTNode(key)
        if key < root.key: root.left = self._inserir(root.left, key)
        else: root.right = self._inserir(root.right, key)
        return root

    def buscar(self, key):
        # Busca elemento na arvore
        return self._buscar(self.root, key)

    def _buscar(self, root, key):
        if not root or root.key == key: return root
        if key < root.key: return self._buscar(root.left, key)
        return self._buscar(root.right, key)

    def remover(self, key):
        # Remove valor da BST
        self.root = self._remover(self.root, key)

    def _remover(self, root, key):
        if not root: return root
        if key < root.key: root.left = self._remover(root.left, key)
        elif key > root.key: root.right = self._remover(root.right, key)
        else:
            if not root.left: return root.right
            if not root.right: return root.left
            temp = self._min_val_node(root.right)
            root.key = temp.key
            root.right = self._remover(root.right, temp.key)
        return root

    def _min_val_node(self, node):
        curr = node
        while curr.left: curr = curr.left
        return curr

    def altura(self):
        # Retorna altura da arvore
        return self._altura(self.root)

    def _altura(self, root):
        if not root: return -1
        return 1 + max(self._altura(root.left), self._altura(root.right))

In [ ]:
# Medicoes para N = 100, 1000 e 10000
for n in [100, 1000, 10000]:
    bst = BST()
    dados = [random.randint(1, 100000) for _ in range(n)]

    # Mede insercoes
    t_ini = time.time()
    for d in dados: bst.inserir(d)
    t_ins = time.time() - t_ini

    # Mede buscas
    t_ini = time.time()
    for d in dados[:50]: bst.buscar(d)
    t_bus = time.time() - t_ini

    print(f"N={n:5} | Inserção: {t_ins:.6f}s | Busca (50 itens): {t_bus:.6f}s")

### Grupo 2 & 3: Computação Paralela com OpenMP e BST
Demonstrações de soma de vetores, produto escalar paralelo e maior elemento em BST.

In [ ]:
%%writefile openmp_tp3.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <omp.h>

// Ex 6: Soma de vetores
void soma_vetores(const std::vector<double>& a, const std::vector<double>& b, std::vector<double>& c) {
    #pragma omp parallel for
    for (size_t i = 0; i < a.size(); i++) {
        c[i] = a[i] + b[i];
    }
}

// Ex 7: Produto escalar com reduction
double produto_escalar(const std::vector<double>& a, const std::vector<double>& b) {
    double res = 0.0;
    #pragma omp parallel for reduction(+:res)
    for (size_t i = 0; i < a.size(); i++) {
        res += a[i] * b[i];
    }
    return res;
}

// Ex 10 e 11: Contagem de primos com OpenMP
bool eh_primo(int n) {
    if (n <= 1) return false;
    for (int i = 2; i * i <= n; i++) {
        if (n % i == 0) return false;
    }
    return true;
}

int contar_primos(int N) {
    int cnt = 0;
    #pragma omp parallel for reduction(+:cnt)
    for (int i = 2; i <= N; i++) {
        if (eh_primo(i)) cnt++;
    }
    return cnt;
}

int main() {
    int N = 100000;
    auto t1 = std::chrono::high_resolution_clock::now();
    int primos = contar_primos(N);
    auto t2 = std::chrono::high_resolution_clock::now();
    std::cout << "Primos ate " << N << ": " << primos << " em "
              << std::chrono::duration_cast<std::chrono::milliseconds>(t2 - t1).count() << " ms" << std::endl;
    return 0;
}

In [ ]:
# Compila e roda testes de OpenMP e BST
!g++ -O3 -fopenmp openmp_tp3.cpp -o openmp_tp3
!./openmp_tp3

### Grupo 4: Árvores de Prefixo (Trie IPv4 e LPM)
Implementação de árvore de prefixos e mecanismo de Longest Prefix Match (LPM) para roteamento IP.

In [ ]:
class TrieNodeIP:
    def __init__(self):
        self.children = {} # Inicializa filhos
        self.is_prefix = False
        self.prefix_str = None

class TrieIP:
    def __init__(self):
        self.root = TrieNodeIP()

    def _ip_to_bits(self, ip, cidr):
        parts = [int(p) for p in ip.split('.')]
        bits = "".join(f"{p:08b}" for p in parts)
        return bits[:cidr]

    def inserir(self, ip, cidr):
        # Insere prefixo na Trie
        bits = self._ip_to_bits(ip, cidr)
        curr = self.root
        for b in bits:
            if b not in curr.children:
                curr.children[b] = TrieNodeIP()
            curr = curr.children[b]
        curr.is_prefix = True
        curr.prefix_str = f"{ip}/{cidr}"

    def longest_prefix_match(self, ip):
        # Busca melhor correspondencia LPM
        bits = self._ip_to_bits(ip, 32)
        curr = self.root
        best_match = None
        for b in bits:
            if b in curr.children:
                curr = curr.children[b]
                if curr.is_prefix:
                    best_match = curr.prefix_str
            else:
                break
        return best_match

# Testando o LPM
trie_ip = TrieIP()
trie_ip.inserir("192.168.0.0", 16)
trie_ip.inserir("192.168.1.0", 24)

print("Busca 192.168.1.10:", trie_ip.longest_prefix_match("192.168.1.10"))
print("Busca 192.168.2.100:", trie_ip.longest_prefix_match("192.168.2.100"))